In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pandas_ta as ta
import math
from tqdm import tqdm
import gc
import time
import json
from pprint import pprint
# import pandas_ta
# import talib
import pickle
# from position_tools import calculate_trades, calculate_positions, count_since_last_signal

# from pklibs import *

import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import talib

# from pklib.strategy import *
from pklib.utilities import *
from pklib.pkindicators import calculate_zigzag
from pklib.indicators import *

# from pklib.rl import *

##################################
### IMPORTANT
# pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm5.7

In [2]:
import dotenv
import os

# Reload the variables in your '.env' file (override the existing variables)
dotenv.load_dotenv(".env", override=True)

# 'MY_VAR' is refreshed now
print('HIP_VISIBLE_DEVICES = ', os.environ.get('HIP_VISIBLE_DEVICES')) # MY_VAR = HELLO_BOB

HIP_VISIBLE_DEVICES =  0


In [3]:

import gymnasium as gym  # Use Gymnasium instead of gym
from gymnasium.spaces import Discrete, Box
from gymnasium import register

In [4]:
# import numpy as np

# # Example PnL array
# pnl = np.array([0.01, 0.02, -0.01, 0.03, -0.02, 0.04])  # Example returns

# # Risk-free rate (set to 0 for simplicity, or use a small number like 0.01 for 1%)
# risk_free_rate = 0.0

# # Calculate daily returns from pnl
# returns = pnl

# 1. Sharpe Ratio
def sharpe_ratio(returns, risk_free_rate=0.0):
    excess_returns = returns - risk_free_rate
    return np.mean(excess_returns) / np.std(excess_returns)

# 2. Sortino Ratio
def sortino_ratio(returns, risk_free_rate=0.0):
    excess_returns = returns - risk_free_rate
    downside_returns = np.where(returns < 0, returns, 0)  # Only consider negative returns
    downside_deviation = np.std(downside_returns)
    return np.mean(excess_returns) / downside_deviation if downside_deviation != 0 else np.inf

def calculate_drawdowns(cumulative_returns):
    cum_rets = cumulative_returns #np.cumsum(rets)
    cum_max = np.maximum.accumulate(cum_rets)
    drawdowns = cum_rets - cum_max
    drawdowns = np.clip(drawdowns, a_min=None, a_max=0)
    
    max_drawdown = np.min(drawdowns)  # Maximum drawdown
    return drawdowns, max_drawdown


# 3. Calmar Ratio
def calmar_ratio(returns,cumulative_returns):
    # cumulative_returns = np.cumprod(1 + returns) - 1
    drawdowns, max_drawdown = calculate_drawdowns(cumulative_returns)
    mean_return = np.mean(returns) #* 252  # Assuming daily returns
    return mean_return / max_drawdown if max_drawdown != 0 else np.inf

# Calculate the ratios
# sharpe = sharpe_ratio(returns, risk_free_rate)
# sortino = sortino_ratio(returns, risk_free_rate)
# calmar = calmar_ratio(returns)

# # Output the ratios
# print(f"Sharpe Ratio: {sharpe:.4f}")
# print(f"Sortino Ratio: {sortino:.4f}")
# print(f"Calmar Ratio: {calmar:.4f}")


In [5]:
# rets = [.1,.15,.11,-0.2,.1,.5,-.1,-.2,-.4,.2]
# cum_rets = np.cumsum(rets)
# drawdowns, max_drawdown = calculate_drawdowns(cum_rets)
# # cum_max = np.maximum.accumulate(cum_rets)
# # drawdowns = cum_rets - cum_max
# # drawdowns = np.clip(drawdowns, a_min=None, a_max=0)
# # drawdowns = np.clip(cum_rets - cum_max, a_min=None, a_max=0)
# # plt.plot(cum_rets)
# plt.plot(drawdowns)
# # calculate_drawdowns(rets)
# # cum_rets

In [6]:


def split_dataframes(dataframes, train_size=0.7):
    """
    Split the input dataframes into training and testing sets.

    Parameters:
    dataframes (list): List of pandas DataFrames to split.
    train_size (float): Proportion of the data to use for training (default is 0.7).

    Returns:
    tuple: (train_dfs, test_dfs) Lists of DataFrames for training and testing.
    """
    train_dfs = []
    test_dfs = []

    for df in dataframes:
        # Calculate the split index
        split_index = int(len(df) * train_size)
        
        # Split the DataFrame
        train_df = df.iloc[:split_index]
        test_df = df.iloc[split_index:]
        
        # Append to the respective lists
        train_dfs.append(train_df)
        test_dfs.append(test_df)

    return train_dfs, test_dfs



In [7]:

class ReplayBuffer(object):
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape),
                                     dtype=np.float32)
        self.new_state_memory = np.zeros((self.mem_size, *input_shape),
                                         dtype=np.float32)

        self.action_memory = np.zeros(self.mem_size, dtype=np.int64)
        self.reward_memory = np.zeros(self.mem_size, dtype=np.float32)
        self.terminal_memory = np.zeros(self.mem_size, dtype=bool)

    def store_transition(self, state, action, reward, state_, done):
        index = self.mem_cntr % self.mem_size
        self.state_memory[index] = state
        self.new_state_memory[index] = state_
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.terminal_memory[index] = done
        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, batch_size, replace=False)

        states = self.state_memory[batch]
        actions = self.action_memory[batch]
        rewards = self.reward_memory[batch]
        states_ = self.new_state_memory[batch]
        terminal = self.terminal_memory[batch]

        return states, actions, rewards, states_, terminal


def plot_learning_curve(x, scores, epsilons, filename, lines=None):
    fig=plt.figure()
    ax=fig.add_subplot(111, label="1")
    ax2=fig.add_subplot(111, label="2", frame_on=False)

    ax.plot(x, epsilons, color="C0")
    ax.set_xlabel("Training Steps", color="C0")
    ax.set_ylabel("Epsilon", color="C0")
    ax.tick_params(axis='x', colors="C0")
    ax.tick_params(axis='y', colors="C0")

    N = len(scores)
    running_avg = np.empty(N)
    for t in range(N):
	    running_avg[t] = np.mean(scores[max(0, t-20):(t+1)])

    ax2.scatter(x, running_avg, color="C1")
    ax2.axes.get_xaxis().set_visible(False)
    ax2.yaxis.tick_right()
    ax2.set_ylabel('Score', color="C1")
    ax2.yaxis.set_label_position('right')
    ax2.tick_params(axis='y', colors="C1")

    if lines is not None:
        for line in lines:
            plt.axvline(x=line)

    plt.savefig(filename)


In [8]:
import os
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

In [9]:
import torch
import torch.nn as nn
from torch.nn.utils.weight_norm import weight_norm


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [10]:
d = T.device('hip')
# T.hip

In [11]:

class DuelingTCQNetwork(nn.Module):
    def __init__(self, lr, n_actions, name, input_dims, chkpt_dir, kernel_size: int = 2, dropout: float = 0.2, fc_layers=[256, 128], num_channels=[16,8,8]):
        super(DuelingTCQNetwork, self).__init__()

        self.checkpoint_dir = chkpt_dir
        self.checkpoint_file = os.path.join(self.checkpoint_dir, name)

        # self.conv1 = nn.Conv2d(input_dims[0], 32, 8, stride=4)
        # self.conv2 = nn.Conv2d(32, 64, 4, stride=2)
        # self.conv3 = nn.Conv2d(64, 64, 3, stride=1)
        self.tcn = TemporalConvNet(input_dims[0], num_channels=num_channels, dropout=dropout, kernel_size=kernel_size )

        fc_input_dims = self.calculate_conv_output_dims(input_dims)
        
        self.fc_layers = nn.ModuleList()
        prev_dim = fc_input_dims
        for layer_size in fc_layers:
            self.fc_layers.append(nn.Linear(prev_dim, layer_size))
            prev_dim = layer_size

        # Output layers for Value and Advantage streams
        self.V = nn.Linear(prev_dim, 1)
        self.A = nn.Linear(prev_dim, n_actions)

        self.optimizer = optim.RMSprop(self.parameters(), lr=lr)
        self.loss = nn.MSELoss()
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        # self.device = T.device('hip:0')
        self.to(self.device)


    def calculate_conv_output_dims(self, input_dims):
        state = T.zeros(1, *input_dims)
        # dims = self.conv1(state)
        # dims = self.conv2(dims)
        # dims = self.conv3(dims)
        dims = self.tcn(state)
        # print(f'dims.shape: {dims.shape}')
        # print(f'dims.size(): {dims.size()}')
        # return dims[1:].item()
        return int(np.prod(dims.size()))

    
    def forward(self, state):
        conv = self.tcn(state)
        conv_state = conv.view(conv.size()[0], -1)

        x = conv_state
        for fc in self.fc_layers:
            x = F.relu(fc(x))

        V = self.V(x)
        A = self.A(x)

        return V, A

    def save_checkpoint(self):
        print('... saving checkpoint ...')
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        print('... loading checkpoint ...')
        self.load_state_dict(T.load(self.checkpoint_file))


In [12]:

class DuelingDDTCQNAgent(object):
    def __init__(self, gamma, epsilon, lr, n_actions, input_dims,
                 mem_size, batch_size, eps_min=0.01, eps_dec=5e-7,
                 replace=1000, kernel_size: int = None, dropout: float = 0.2,  algo=None, env_name=None, chkpt_dir='tmp/dqn', fc_layers=None, num_channels=None):
        self.gamma = gamma
        self.epsilon = epsilon
        self.lr = lr
        self.n_actions = n_actions
        self.input_dims = input_dims
        self.batch_size = batch_size
        self.eps_min = eps_min
        self.eps_dec = eps_dec
        self.replace_target_cnt = replace
        self.algo = algo
        self.env_name = env_name
        self.chkpt_dir = chkpt_dir
        self.action_space = [i for i in range(n_actions)]
        self.learn_step_counter = 0
        self.kernel_size = kernel_size
        self.dropout = dropout

        self.memory = ReplayBuffer(mem_size, input_dims, n_actions)

        self.q_eval = DuelingTCQNetwork(self.lr, self.n_actions,
                        input_dims=self.input_dims,
                        name=self.env_name+'_'+self.algo+'_q_eval',
                        chkpt_dir=self.chkpt_dir, 
                        kernel_size=kernel_size, dropout=dropout,
                        fc_layers=fc_layers, num_channels=num_channels)
        self.q_next = DuelingTCQNetwork(self.lr, self.n_actions,
                        input_dims=self.input_dims,
                        name=self.env_name+'_'+self.algo+'_q_next',
                        chkpt_dir=self.chkpt_dir, 
                        kernel_size=kernel_size, dropout=dropout,
                        fc_layers=fc_layers, num_channels=num_channels)
        
        # self.q_eval = DuelingDeepQNetwork(self.lr, self.n_actions,
        #                 input_dims=self.input_dims,
        #                 name=self.env_name+'_'+self.algo+'_q_eval',
        #                 chkpt_dir=self.chkpt_dir)
        # self.q_next = DuelingDeepQNetwork(self.lr, self.n_actions,
        #                 input_dims=self.input_dims,
        #                 name=self.env_name+'_'+self.algo+'_q_next',
        #                 chkpt_dir=self.chkpt_dir)

    def store_transition(self, state, action, reward, state_, done):
        self.memory.store_transition(state, action, reward, state_, done)

    def sample_memory(self):
        state, action, reward, new_state, done = \
                                self.memory.sample_buffer(self.batch_size)

        states = T.tensor(state, dtype=T.float32).to(self.q_eval.device)
        rewards = T.tensor(reward, dtype=T.float32).to(self.q_eval.device)
        dones = T.tensor(done, dtype=T.bool).to(self.q_eval.device)
        actions = T.tensor(action, dtype=T.int64).to(self.q_eval.device)
        states_ = T.tensor(new_state, dtype=T.float32).to(self.q_eval.device)

        return states, actions, rewards, states_, dones

    def choose_action(self, observation, is_training=True):
        if not is_training or (np.random.random() > self.epsilon):
            state = np.array([observation], copy=False, dtype=np.float32)
            state_tensor = T.tensor(state, dtype=T.float32).to(self.q_eval.device)
            _, advantages = self.q_eval.forward(state_tensor)

            action = T.argmax(advantages).item()
        else:
            action = np.random.choice(self.action_space)

        return action

    def replace_target_network(self):
        if self.replace_target_cnt is not None and \
           self.learn_step_counter % self.replace_target_cnt == 0:
            self.q_next.load_state_dict(self.q_eval.state_dict())

    def decrement_epsilon(self):
        self.epsilon = self.epsilon - self.eps_dec \
                           if self.epsilon > self.eps_min else self.eps_min

    def learn(self):
        if self.memory.mem_cntr < self.batch_size:
            return

        self.q_eval.optimizer.zero_grad()

        self.replace_target_network()

        states, actions, rewards, states_, dones = self.sample_memory()
        indices = np.arange(self.batch_size)

        V_s, A_s = self.q_eval.forward(states)
        V_s_, A_s_ = self.q_next.forward(states_)

        V_s_eval, A_s_eval = self.q_eval.forward(states_)

        q_pred = T.add(V_s,
                        (A_s - A_s.mean(dim=1, keepdim=True)))[indices, actions]

        q_next = T.add(V_s_, (A_s_ - A_s_.mean(dim=1, keepdim=True)))

        q_eval = T.add(V_s_eval, (A_s_eval - A_s_eval.mean(dim=1,keepdim=True)))

        max_actions = T.argmax(q_eval, dim=1)
        q_next[dones] = 0.0

        q_target = rewards + self.gamma*q_next[indices, max_actions]

        loss = self.q_eval.loss(q_target, q_pred).to(self.q_eval.device)
        loss.backward()
        self.q_eval.optimizer.step()
        self.learn_step_counter += 1

        self.decrement_epsilon()

    def save_models(self):
        self.q_eval.save_checkpoint()
        self.q_next.save_checkpoint()

    def load_models(self):
        self.q_eval.load_checkpoint()
        self.q_next.load_checkpoint()


In [13]:
def make_env(env_name, env_config, shape=(84,84,1), repeat=4, clip_rewards=False,
             no_ops=0, fire_first=False):
    env = gym.make(env_name,env_config=env_config)
    # env = RepeatActionAndMaxFrame(env, repeat, clip_rewards, no_ops, fire_first)
    # env = PreprocessFrame(shape, env)
    # env = StackFrames(env, repeat)

    return env

In [14]:
class TradingEnv(gym.Env):
    def __init__(self, env_config):
        self.env_config = env_config
        
        self.df = env_config['df']
        self.logprice = env_config['logprice']
        self.lookback_window_size = env_config['lookback_window_size']
        self.trading_mode = env_config.get('trading_mode', 'both')
        self.trading_fee = env_config.get('trading_fee', np.log1p(0.001))
        self.verbosity = env_config.get('verbosity', 0)
        self.reward_key = env_config.get('reward_key', 'sharpe')
        self.reward_aggregation = env_config.get('reward_aggregation', 'episode')
        self.reward_nsteps = env_config.get('reward_nsteps', 1000)
        
        self.current_step = 0
        self.entry_logprice = None
        self.cum_ret = 0
        self.num_trades = 0
        self.position = 0  # 1 = long, 0 = no position, -1 = short
        self.rets = []

        # Define action space based on trading mode
        if self.trading_mode == 'long_only':
            self.action_space = Discrete(2)  # 0 = hold, 1 = long
        elif self.trading_mode == 'short_only':
            self.action_space = Discrete(2)  # 0 = hold, 1 = short
        else:  # 'both'
            self.action_space = Discrete(3)  # 0 = hold, 1 = long, 2 = short, 3 = close position
        # self.action_space = Box(low=-np.inf, high=np.inf, shape=obs_space_shape, dtype=np.float32)
        # Observation space: OHLCV + technical indicators + entry price + position (flattened lookback period + 2 additional features)
        obs_space_shape = (self.lookback_window_size , len(self.df.columns) + 1)
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=obs_space_shape, dtype=np.float32)
        
    def reset(self, seed=None, options=None):
        # Ensure enough data for the lookback window
        self.current_step = self.lookback_window_size

        self.entry_logprice = None
        self.position = 0  # 1 = long, 0 = no position, -1 = short
        self.entry_step = None
        self.cum_ret = 0  
        self.rets = []
        self.num_trades = 0

        # Get the starting index for the lookback window (ensure it's valid)
        start_index = max(0, self.current_step - self.lookback_window_size)
        frame = self.df.iloc[start_index:self.current_step].values

        # Pad the history if there are fewer observations than the lookback size
        if len(frame) < self.lookback_window_size:
            padding = np.zeros((self.lookback_window_size - len(frame), len(self.df.columns)))
            frame = np.vstack((padding, frame))
            
        frame = np.hstack([frame, np.full((frame.shape[0], 1), self.position)])

        # Return the initial flattened observation (1D array)
        # The 'info' dictionary is returned for compatibility with Gym environments
        # return np.hstack([initial_observation.flatten()]).astype(np.double), {'cumulative_pnl': self.cumulative_pnl}
        info = {'cum_ret': self.cum_ret, 'num_trades': self.num_trades}
        return frame, info

    def close_position(self, current_logprice):
        ret = self.position * (current_logprice - self.entry_logprice)
        ret -= 2*self.trading_fee
        self.rets.append(ret)
        self.cum_ret += ret  # Accumulate PnL
        self.num_trades += 1
        trade_info = {
            'entry_step': self.entry_step,
            'exit_step': self.current_step,
            'position': self.position,
            'ret': ret
        }
        if self.verbosity >= 2:
            print(f"Closed {self.position} position. Return: {ret}")
        self.position = 0
        self.entry_step = None
        
        return trade_info
    
    def step(self, action):
        current_logprice = self.logprice.iloc[self.current_step]
        reward = 0
        pnl = 0
        trade_info = None
        term_info = None

        if self.trading_mode == 'short_only' and action == 1 : action = -1
        if self.trading_mode == 'both' and action == 2: action = -1
        
        if self.verbosity >= 2:
            print(f"Step: {self.current_step}, Action: {action}, Position: {self.position}, Cum Ret: {self.cum_ret}")
            
        if self.position == 0:
            open_position = (action == 1 and self.trading_mode in ['long_only', 'both']) or (action == -1 and self.trading_mode in ['short_only', 'both'])
        else:
            open_position = self.position != action and action != 0
        # open_position = ((self.position == 0) and ((action == 1 and self.trading_mode in ['long_only', 'both']) or (action == -1 and self.trading_mode in ['short_only', 'both']))) \
        #     or (self.position != 0 and self.position != action)
            
        if self.position != 0 and self.position != action:
            trade_info = self.close_position(current_logprice)
            self.position = 0
            
        if open_position:
            reward = - self.env_config.get('overtrade_penalty', 0)
            self.entry_logprice = current_logprice
            self.position = action
            self.entry_step = self.current_step
            if self.verbosity >= 1:
                print(f"Opened {action} position at {current_logprice}")
                
        # elif (self.position != 0) and (action == 0):
        #     trade_info = self.close_position(current_logprice)
            
        # Move to the next step
        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1  # End of data
        truncated = False


        if terminated and self.position != 0:
            trade_info = self.close_position(current_logprice)
            
        # Handle termination case with open position
        if terminated or ((self.reward_aggregation == 'nsteps') and (self.current_step % self.reward_nsteps == 0)):
                    

            returns = np.array(self.rets)
            cum_returns = returns.cumsum()
            pct_returns = np.expm1(returns)
            pct_cum_returns = np.expm1(cum_returns)
            sharpe, sortino, calmar = 0,0,0
            drawdowns, max_drawdown = [], 0
            if len(returns) > 0:
                drawdowns, max_drawdown = calculate_drawdowns(cum_returns)
                sharpe = sharpe_ratio(pct_returns)
                sortino = sortino_ratio(pct_returns)
                calmar = calmar_ratio(pct_returns, pct_cum_returns)
                
            term_info = {
                'num_trades': self.num_trades,
                'cum_ret': self.cum_ret,
                'sharpe': sharpe,
                'sortino': sortino,
                'calmar': calmar,
                'max_drawdown': max_drawdown,
            }
            reward = term_info[self.reward_key]
            
            self.cum_ret = 0
            self.num_trades = 0
            self.rets = []
        else:
            reward = pnl

        # Get the next observation
        obs = self._next_observation()

        # Return observation, reward, termination info, and trade details
        info = {
            'position': self.position,
            'cum_ret': self.cum_ret,
            'num_trades': self.num_trades,
            'trade_info': trade_info,
            'term_info': term_info
        }

        return obs, reward, terminated, truncated, info

    def _next_observation(self):
        # Handle edge cases where the current step is less than the lookback window size
        start_index = max(0, self.current_step - self.lookback_window_size)
        frame = self.df.iloc[start_index:self.current_step]

        # If the lookback window is smaller than the full lookback size, pad with earlier rows
        if len(frame) < self.lookback_window_size:
            padding = np.zeros((self.lookback_window_size - len(frame), len(self.df.columns)))
            frame = np.vstack((padding, frame.values))
        else:
            frame = frame.values

        frame = np.hstack([frame, np.full((frame.shape[0], 1), self.position)])
        # Return flattened lookback window and append position and entry price
        # return np.hstack([frame.flatten()]).astype(np.double)
        return frame
    
    def render(self, mode="human"):
        # Print relevant information about the environment (for debugging purposes)
        print(f"Step: {self.current_step}, #Positions: {self.num_trades}, Cum Ret: {self.cum_ret:.4f}")



In [15]:
fib_levels = np.array([-1.0, -0.786, -0.618, -0.5, -0.382, -0.236, 0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0, 1.236, 1.5, 1.618, 1.786, 2.0, 2.236, 2.382, 2.5, 2.628, 2.786, 3, 3.382, 3.618, 4, 5])

fib_columns = [f'fib({fib})' for fib in fib_levels]
# fib_columns 

def calculate_fib_levels(highs, lows, directions, fib_levels=fib_levels):
    # Ensure highs, lows, and directions are numpy arrays
    highs = np.asarray(highs)
    lows = np.asarray(lows)
    directions = np.asarray(directions)

    # Flip highs and lows based on direction (-1 means flip)
    adjusted_highs = np.where(directions == 1, highs, lows)
    adjusted_lows = np.where(directions == 1, lows, highs)

    # Calculate the difference between adjusted high and low
    diff = adjusted_highs - adjusted_lows

    # Calculate the Fibonacci levels by applying the levels to the differences
    fib_matrix = np.outer(diff, fib_levels)
    
    # Calculate the actual levels by adding them to the low (base) level
    fib_levels_array = adjusted_lows[:, np.newaxis] + fib_matrix

    return fib_levels_array


def get_fibs_df(data, epsilon):
    """
    Plot close prices with highs and lows markers, and Fibonacci levels.

    Parameters:
    - data: DataFrame containing OHLCV data with a 'close' column.
    - epsilon: Epsilon value used for the ZigZag calculation.
    - window: Optional tuple (start, end) to define the plotting window.
    """

    # Call the calculate_zigzag function from the C module
    high_low_markers, turning_points = calculate_zigzag(data['close'].values, epsilon=epsilon)

    # Store the results back into the DataFrame for easier plotting
    data['HighLowMarkers'] = high_low_markers
    data['TurningPoints'] = turning_points

    # Get the indices of highs and lows
    highs_idx = data.index[data['HighLowMarkers'] == 1]
    lows_idx = data.index[data['HighLowMarkers'] == -1]

    running_highs = (np.where(high_low_markers == 1, 1, np.nan) * data['close']).ffill()
    running_lows = (np.where(high_low_markers == -1, 1, np.nan) * data['close']).ffill()

    running_highs_idx = pd.Series(np.where(high_low_markers == 1, 1, np.nan) * np.arange(len(data))).set_axis(data.index).ffill()
    running_lows_idx = pd.Series(np.where(high_low_markers == -1, 1, np.nan) * np.arange(len(data))).set_axis(data.index).ffill()

    # Combine the indices and sort them
    # turning_points_idx = sorted(highs_idx.union(lows_idx))

    # Assuming calculate_fib_levels is defined elsewhere in your code
    fibs = calculate_fib_levels(running_lows, running_highs, ((running_highs_idx > running_lows_idx) * 2 - 1))
    df_fibs = pd.DataFrame(fibs, columns=fib_columns, index=data.index)
    return df_fibs


In [16]:

# df

In [17]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler


# params = {'conversion_periods': 110, 'base_periods': 301, 'lagging_span2_periods': 250, 'displacement': 180}
params = {'conversion_periods': 20, 'base_periods': 60, 'lagging_span2_periods': 120, 'displacement': 30}
# Parameters
exchange = 'binance'; asset = 'BTC'; quote = 'USDT'; nhours = 8
train_size= 0.5

# lookback_window_size = int(24 // nhours)*7*2
lookback_window_size = 1

df = load_candles(exchange, asset, quote, '1h').ffill().bfill()
df = df.resample(f'{nhours}H').agg({'open': 'first','high': 'max','low': 'min','close': 'last','volume': 'sum'}).ffill().bfill()
df.drop('volume', axis=1)

# add_ichimoku_cloud_indicator(df, params)
# add_bollinger_bands(df, periods=[9,14,21,50,100], multiplier=2)
# add_rsi_columns(df,periods=[9,14,21,50,100])
# add_adx_columns(df,periods=[9,14,21,50,100])
# add_mom_columns(df,periods=[3,5,7,9,14,21,50,100])
# add_ema_columns(df,periods=[9,14,21,50,100])
# add_sma_columns(df,periods=[9,14,21,50,100])
# add_std_columns(df,periods=[7,9,7,14,21])
# log_price_over_ma_columns(df,periods=[3,5,9,14,21,50])
# add_sma_columns(df,periods=[14,21,50,100,200])
# add_std_columns(df,periods=[14,21,50,100,200])
# add_rsi_columns(df,periods=[4,8,16,32,64])
# add_rolling_max_columns(df,periods=[9,14,21,50,120], column="close")
# add_rolling_min_columns(df,periods=[9,14,21,50,120], column="close")
# add_rolling_max_columns(df,periods=[9,14,21,50], column="high")
# add_rolling_min_columns(df,periods=[9,14,21,50], column="low")

# add_shifted_columns(df, periods=[14,21,50,100,200,400], columns=['open','high','low','close'], suffix="_SH")
# periods=[4,8,16,32,64,128]
periods=[1,2,3,4]
add_shifted_columns(df, periods=periods, columns=['open','high','low','close'], suffix="_SH")

df_fibs = get_fibs_df(df, epsilon=0.2)
df = df.join(df_fibs)

df = df.dropna()


# df = df['2021':]#.iloc[:10000]
print(f'DF shape: {df.shape}')
df_logprice = df.close.apply(np.log).ffill().bfill()

# Rescale / Normalize features
scaler = StandardScaler()
# df = df.apply(np.log)
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

(train_df_features, train_df_log_prices), (test_df_features, test_df_log_prices) = split_dataframes([df, df_logprice], train_size=train_size)
df

n_train,n_features = train_df_features.shape
feat_multiples = math.ceil(df.shape[1] / n_features)

# feat_multiples = len(periods)+1
# n_features = df.shape[1] // feat_multiples


DF shape: (7688, 51)


In [18]:

# from gymnasium.envs.registration import register

register(
    id='TradingEnv-v0',   # Unique id for the environment
    entry_point=TradingEnv,
    # entry_point='__main__:TradingEnv',  # Path to the custom environment
    # max_episode_steps=200,  # Optional: max number of steps per episode
)

# Step 3: Create the environment using gymnasium.make
# env = gym.make('TradingEnv-v0')



In [19]:
# env.cum_pnl

### TRAING

In [23]:

# env = train_env = TradingEnv(train_env_config)

trading_mode = 'both'# Wrap the environment
train_env_config = {
    'df': train_df_features,  # Your training features DataFrame
    'logprice': train_df_log_prices,  # Your log prices
    'lookback_window_size': lookback_window_size,
    'trading_mode': trading_mode,  # or 'short_only', 'both'
    'trading_fee': 0.0,
    'reward_aggregation': 'steps',
    'reward_nsteps': 500,
    'reward_key': 'sharpe',
    'overtrade_penalty': 0.02,
    'verbosity': 0
}
env = make_env('TradingEnv-v0',env_config=train_env_config)

print(f'train_df_features.shape: {train_df_features.shape}')
observation_shape = env.observation_space.shape
print("Observation shape:", observation_shape)

best_score = -np.inf
load_checkpoint = False
n_games = 100

agent = DuelingDDTCQNAgent(gamma=0.99, epsilon=1, lr=0.0001,
                    input_dims=(env.observation_space.shape),
                    # input_dims=(feat_multiples,n_features),
                    n_actions=env.action_space.n, mem_size=10000, eps_min=0.1,
                    batch_size=32, replace=5000, eps_dec=1e-5,
                    chkpt_dir='models/', algo='DuelingDTCQNAgent',
                    env_name='TradingEnv-v0',
                    dropout=0.2,
                    kernel_size=4,
                    # num_channels=[n_features//2,n_features//2,n_features//4] #* feat_multiples
                    # fc_layers=[2*n_features,2*n_features],
                    num_channels=[n_features//2,n_features//4],
                    fc_layers=[],
                    # fc_layers=[256,256],
                    )
n_steps = 0; score = 0; total_score=0;
scores, steps_array, num_trades, cum_rets, term_infos = [], [], [], [], []

# agent.epsilon = 1
# agent.lr = 0.0001

eps_history = []

if load_checkpoint:
    agent.load_models()

fname = agent.algo + '_' + agent.env_name + '_lr' + str(agent.lr) +'_' \
        + str(n_games) + 'games'
figure_file = 'plots/' + fname + '.png'
# step = 0
# step_chk = 1000
# lr = lr_start = 0.0003;lr_min=0.00005;lr_delta=(lr_start-lr_min)/1000;
for i in range(n_games):
    done = False
    observation, _ = env.reset()
    # observation = observation.reshape(feat_multiples, n_features)    
    # observation = observation.T
    
    while not done:
        # lr -= lr_delta
        # if lr < lr_min:
        #     lr = lr_start
        # lr = lr_min + np.random.random() * (lr_start - lr_min)
        # agent.lr = lr
        lr = agent.lr

        action = agent.choose_action(observation)
        observation_, reward, done, _, info = env.step(action)
        # observation_ = observation_.reshape(feat_multiples, n_features)    
        # observation_ = observation_.T
        score += reward
        total_score += reward

        if not load_checkpoint:
            agent.store_transition(observation, action,
                                    reward, observation_, int(done))
            agent.learn()
        observation = observation_
        n_steps += 1
        
        # if n_steps % step_chk == 0:

        if info['term_info'] is not None:
            term_info = info['term_info']
            
            term_infos.append({
                'score': score,
                'total_score': total_score,
                **term_info
            })
            eps_history.append(agent.epsilon)
            cum_rets.append(term_info['cum_ret'])
            num_trades.append(term_info['num_trades'])
            scores.append(score)
            steps_array.append(n_steps)
        
            avg_cum_ret = np.sum(cum_rets[-10:])
            tot_cum_ret = np.sum(cum_rets)
            print('')
            # print(term_info)
            print(f'{"Epsd":<4} | {"lr":<8} | {"TotScr":<8} | {"Score":<6} | {"Rewrd":<6} | {"TCmRet":<6} | {"ACmRet":<6} | {"CmRet":<6} | {"#Trds":<4} | {"Epsiln":<4} | {"Steps":<5} | {"Sharpe":<6} | {"Srtino":<6} | {"Calmar":<6} | {"MaxDDN":<6} |')
            print(f'{i:>4} | {lr:>8.5f} | {total_score:>8.1f} | {score:>6.2f} | {reward:>6.2f} | {(tot_cum_ret):>6.2f} | {(avg_cum_ret):>6.2f} | {term_info["cum_ret"]:>6.2f} | {num_trades[-1]:>5} | {agent.epsilon:>4.4f} | {n_steps:>5} | {term_info["sharpe"]:>6.2f} | {term_info["sortino"]:>6.2f} | {term_info["calmar"]:>6.2f} | {np.expm1(term_info["max_drawdown"]):>6.2f} |')
            
            score = 0
    # avg_score = np.mean(scores[-10:])
    # print('episode: ', i,'score: %.2f\t' % score, 'pnl: %.2f\t' % np.expm1(score),
    #         # ' average score %.1f' % avg_score, 
    #     'best score: %.2f\t' % best_score,
    #     'epsilon: %.2f\t' % agent.epsilon, 'steps: \t'% n_steps)


    if score > best_score:
        if not load_checkpoint:
            agent.save_models()
        best_score = score
    # if avg_score > best_score:
    #     if not load_checkpoint:
    #         agent.save_models()
    #     best_score = avg_score

    # eps_history.append(agent.epsilon)
    if load_checkpoint and n_steps >= 18000:
        break

agent.save_models()
# x = [i+1 for i in range(len(scores))]
plot_learning_curve(steps_array, scores, eps_history, figure_file)


train_df_features.shape: (3844, 51)
Observation shape: (1, 52)

Epsd | lr       | TotScr   | Score  | Rewrd  | TCmRet | ACmRet | CmRet  | #Trds | Epsiln | Steps | Sharpe | Srtino | Calmar | MaxDDN |
   0 |  0.00030 |     -0.0 |  -0.02 |  -0.02 |  -1.96 |  -1.96 |  -1.96 |  1688 | 0.9619 |  3842 |  -0.02 |  -0.03 |   0.00 |  -0.93 |
... saving checkpoint ...
... saving checkpoint ...

Epsd | lr       | TotScr   | Score  | Rewrd  | TCmRet | ACmRet | CmRet  | #Trds | Epsiln | Steps | Sharpe | Srtino | Calmar | MaxDDN |
   1 |  0.00030 |     -0.0 |   0.01 |   0.01 |  -2.37 |  -2.37 |  -0.41 |  1763 | 0.9235 |  7684 |   0.01 |   0.01 |  -0.00 |  -0.81 |

Epsd | lr       | TotScr   | Score  | Rewrd  | TCmRet | ACmRet | CmRet  | #Trds | Epsiln | Steps | Sharpe | Srtino | Calmar | MaxDDN |
   2 |  0.00030 |      0.0 |   0.04 |   0.04 |  -1.05 |  -1.05 |   1.32 |  1685 | 0.8851 | 11526 |   0.04 |   0.07 |  -0.00 |  -0.47 |

Epsd | lr       | TotScr   | Score  | Rewrd  | TCmRet | ACmRet | CmRet 

In [ ]:
agent.save_models()
plot_learning_curve(steps_array, scores, eps_history, figure_file)

In [ ]:
# np.expm1(np.sum(cum_rets))
pd.DataFrame(term_infos).cum_ret.cumsum().apply(np.expm1).plot()

### TESTING

In [ ]:
# net = DuelingTCQNetwork(0,2,'TradingEnv-v0_q_eval', )

In [ ]:

test_env_config = {
    'df': test_df_features,  # Your training features DataFrame
    'logprice': test_df_log_prices,  # Your log prices
    'lookback_window_size': lookback_window_size,
    'trading_mode': trading_mode,  # or 'short_only', 'both'
    'trading_fee': 0,
    'reward_aggregation': 'episode',
    'reward_nsteps': 1,
    'reward_key': 'cum_ret',
    'verbosity': 2
}

# env = train_env = TradingEnv(train_env_config)

env = make_env('TradingEnv-v0',env_config=test_env_config)

print(f'test_df_features.shape: {test_df_features.shape}')
observation_shape = env.observation_space.shape
print("Observation shape:", observation_shape)

best_score = -np.inf
load_checkpoint = True
n_games = 1

fname = agent.algo + '_' + agent.env_name + '_lr' + str(agent.lr) +'_' \
        + str(n_games) + 'games'
figure_file = 'plots/' + fname + '.png'

n_steps = 0
# scores, steps_array, num_trades, cum_rets = [], [], [], []

trade_infos = []
# step = 0
# step_chk = 1000
score = 0; total_score=0;
for i in range(n_games):
    done = False
    observation, _ = env.reset()

    while not done:
        action = agent.choose_action(observation, is_training=False)
        observation_, reward, done, _, info = env.step(action)
        score += reward
        total_score += reward

        observation = observation_
        n_steps += 1
        # print(info)
        # if n_steps % step_chk == 0:

        if info['trade_info'] is not None:
            trade_info = info['trade_info']
            # print(trade_infos)
            trade_infos.append(trade_info)
            score = 0
            
pd.DataFrame(trade_infos).ret.cumsum().apply(np.expm1).plot()

In [ ]:
trade_infos

In [ ]:
info

In [ ]:
250 * nhours / 24